## A notebook to create a bar graph of CTs inside AS

## Install and import libraries

In [167]:
%pip install pandas plotly networkx pydot

import pandas as pd
import requests
from  io import StringIO
from pprint import pprint
import networkx as nx
import plotly.graph_objects as go
import pydot  # required for graphviz layout
from networkx.drawing.nx_pydot import graphviz_layout

Note: you may need to restart the kernel to use updated packages.


## Global settings

In [168]:
hra_pop_version = 'v1.0'
branch = 'main'

output_folder = 'output/ctann-tree'

## Download crosswalks and extract CTs

In [169]:
crosswalk_azimuth = pd.read_csv(
    'https://cdn.humanatlas.io/digital-objects/ctann/azimuth/v1.2/assets/azimuth-crosswalk.csv', skiprows=10)
crosswalk_azimuth

,Organ_Level,Organ_ID,Annotation_Label,Annotation_Label_ID,CL_Label,CL_ID,CL_Match
0,Heart_L2,UBERON:0000948,Adipocyte,AZ:0000001,adipocyte,CL:0000136,skos:exactMatch
1,Heart_L2,UBERON:0000948,Arterial Endothelial,AZ:0000002,endothelial cell of artery,CL:1000413,skos:exactMatch
2,Heart_L2,UBERON:0000948,Atrial Cardiomyocyte,AZ:0000003,regular atrial cardiac myocyte,CL:0002129,skos:exactMatch
3,Heart_L2,UBERON:0000948,B,AZ:0000004,B cell,CL:0000236,skos:exactMatch
4,Heart_L2,UBERON:0000948,Capillary Endothelial,AZ:0000005,capillary endothelial cell,CL:0002144,skos:exactMatch
...,...,...,...,...,...,...,...
759,Kidney,UBERON:0002113,Peritubular Capilary Endothelial,NaN,peritubular capillary endothelial cell,CL:1001033,skos:exactMatch
760,Bone_marrow,UBERON:0002371,CD8 Effector_1,NaN,"effector CD8-positive, alpha-beta T cell:1",CL:0001050,skos:narrowMatch
761,Bone_marrow,UBERON:0002371,CD8 Effector_2,NaN,"effector CD8-positive, alpha-beta T cell:2",CL:0001050,skos:narrowMatch
762,Bone_marrow,UBERON:0002371,CD8 Effector_3,NaN,"effector CD8-positive, alpha-beta T cell:3",CL:0001050,skos:narrowMatch


In [170]:
crosswalk_celltypist = pd.read_csv(
    'https://cdn.humanatlas.io/digital-objects/ctann/celltypist/v1.1/assets/celltypist-crosswalk.csv', skiprows=10)
crosswalk_celltypist

,Organ_Level,Organ_ID,Annotation_Label,Annotation_Label_ID,CL_Label,CL_ID,CL_Match
0,blood_L1,UBERON:0000178,Age-associated B cells,CT:0000001,B cell:age-associated,CL:0000236,skos:narrowMatch
1,blood_L1,UBERON:0000178,C1 non-classical monocytes,CT:0000002,non-classical monocyte:C1,CL:0000875,skos:narrowMatch
2,blood_L1,UBERON:0000178,CD16+ NK cells,CT:0000003,"CD16-positive, CD56-dim natural killer cell, h...",CL:0000939,skos:exactMatch
3,blood_L1,UBERON:0000178,CD16- NK cells,CT:0000004,"CD16-negative, CD56-bright natural killer cell...",CL:0000938,skos:exactMatch
4,blood_L1,UBERON:0000178,Classical monocytes,CT:0000005,classical monocyte,CL:0000860,skos:exactMatch
...,...,...,...,...,...,...,...
888,Small_Intestine,UBERON:0002108,myofibroblast,NaN,myofibroblast cell,CL:0000186,skos:exactMatch
889,Small_Intestine,UBERON:0002108,myofibroblast (RSPO2+),NaN,myofibroblast cell:RSPO2+,CL:0000186,skos:narrowMatch
890,Small_Intestine,UBERON:0002108,pDC,NaN,plasmacytoid dendritic cell,CL:0000784,skos:exactMatch
891,Small_Intestine,UBERON:0002108,venous capillary,NaN,pre-venule capillary cell,CL:4047030,skos:exactMatch


In [171]:
crosswalk_popv = pd.read_csv(
    'https://cdn.humanatlas.io/digital-objects/ctann/popv/v1.2/assets/popv-crosswalk.csv', skiprows=10)
crosswalk_popv

,Organ_Level,Organ_ID,Annotation_Label,Annotation_Label_ID,CL_Label,CL_ID,CL_Match
0,blood,UBERON:0000178,CD141-positive myeloid dendritic cell,PV:0000001,CD141-positive myeloid dendritic cell,CL:0002394,skos:exactMatch
1,blood,UBERON:0000178,"CD4-positive, alpha-beta memory T cell",PV:0000002,"CD4-positive, alpha-beta memory T cell",CL:0000897,skos:exactMatch
2,blood,UBERON:0000178,"CD8-positive, alpha-beta T cell",PV:0000003,"CD8-positive, alpha-beta T cell",CL:0000625,skos:exactMatch
3,blood,UBERON:0000178,"CD8-positive, alpha-beta cytokine secreting ef...",PV:0000004,"CD8-positive, alpha-beta cytokine secreting ef...",CL:0000908,skos:exactMatch
4,blood,UBERON:0000178,T cell,PV:0000005,T cell,CL:0000084,skos:exactMatch
...,...,...,...,...,...,...,...
442,prostate gland,UBERON:0002367,bronchial epithelial cell,NaN,epithelial cell,CL:0000066,skos:narrowMatch
443,thymus,UBERON:0002370,"CD4-positive, CD25-positive, alpha-beta regula...",NaN,"CD4-positive, CD25-positive, alpha-beta regula...",CL:0000792,skos:exactMatch
444,thymus,UBERON:0002370,"CD4-positive, alpha-beta T cell",NaN,"CD4-positive, alpha-beta T cell",CL:0000624,skos:exactMatch
445,bone marrow,UBERON:0002371,"B cell, CD19-positive",NaN,"B cell, CD19-positive",CL:0001201,skos:exactMatch


In [172]:
crosswalk_vccf = pd.read_csv(
    'https://cdn.humanatlas.io/digital-objects/ctann/vccf/v1.0/assets/vccf-crosswalk.csv', skiprows=10)
crosswalk_vccf

,Organ_Level,Organ_ID,Annotation_Label,Annotation_Label_ID,CL_Label,CL_ID,CL_Match
0,bonemarrow-codex-chop_L1,UBERON:0002371,VCCF:0000001,mesenchymal cell,mesenchymal cell,CL:0008019,skos:exactMatch
1,bonemarrow-codex-chop_L1,UBERON:0002371,VCCF:0000002,unknown cell,cell:unknown,CL:0000000,skos:narrowMatch
2,bonemarrow-codex-chop_L1,UBERON:0002371,VCCF:0000003,immune cell,leukocyte,CL:0000738,skos:exactMatch
3,bonemarrow-codex-chop_L1,UBERON:0002371,VCCF:0000004,hematopoeitic precursor cell,hematopoietic precursor cell,CL:0008001,skos:exactMatch
4,bonemarrow-codex-chop_L1,UBERON:0002371,VCCF:0000005,endothelial cell,endothelial cell,CL:0000115,skos:exactMatch
...,...,...,...,...,...,...,...
497,tonsil-codex-stanford_L3,UBERON:0002372,VCCF:0000498,squamous epithelial cell,squamous epithelial cell,CL:0000076,skos:exactMatch
498,tonsil-codex-stanford_L3,UBERON:0002372,VCCF:0000499,stromal cell,stromal cell,CL:0000499,skos:exactMatch
499,tonsil-codex-stanford_L3,UBERON:0002372,VCCF:0000500,t cell,T cell,CL:0000084,skos:exactMatch
500,NaN,NaN,VCCF:0000501,endothelial cell,endothelial cell,CL:0000115,skos:exactMatch


In [173]:
# extract CL IDs and labels with tool
extract = ['CL_ID', 'CL_Label']

# Extract the columns from each DataFrame
az_selected = crosswalk_azimuth[extract].assign(tool='azimuth')
ct_selected = crosswalk_celltypist[extract].assign(tool='celltypist')
popv_selected = crosswalk_popv[extract].assign(tool='popv')
vccf_selected = crosswalk_vccf[extract].assign(tool='vccf')

# Concatenate them into one DataFrame
df_crosswalks_combined = pd.concat(
    [az_selected, ct_selected, popv_selected, vccf_selected], ignore_index=True)

df_crosswalks_combined

,CL_ID,CL_Label,tool
0,CL:0000136,adipocyte,azimuth
1,CL:1000413,endothelial cell of artery,azimuth
2,CL:0002129,regular atrial cardiac myocyte,azimuth
3,CL:0000236,B cell,azimuth
4,CL:0002144,capillary endothelial cell,azimuth
...,...,...,...
2601,CL:0000076,squamous epithelial cell,vccf
2602,CL:0000499,stromal cell,vccf
2603,CL:0000084,T cell,vccf
2604,CL:0000115,endothelial cell,vccf


## Get CT level mapping

In [174]:
url = f'https://raw.githubusercontent.com/x-atlas-consortia/hra-pop/refs/heads/{branch}/output-data/{hra_pop_version}/reports/atlas-ad-hoc/cell-types-level-mapping.csv'

df_cell_types_level_mapping = pd.read_csv(url)
df_cell_types_level_mapping

,cell_label,cell_id,level_1_cell_id,level_1_cell_label,level_2_cell_id,level_2_cell_label
0,cell,http://purl.obolibrary.org/obo/CL_0000000,http://purl.obolibrary.org/obo/CL_0000000,no mapped parent cell,http://purl.obolibrary.org/obo/CL_0000000,no mapped parent cell
1,hematopoietic stem cell,http://purl.obolibrary.org/obo/CL_0000037,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell
2,fibroblast,http://purl.obolibrary.org/obo/CL_0000057,http://purl.obolibrary.org/obo/CL_0002320,connective tissue cell,http://purl.obolibrary.org/obo/CL_0002320,connective tissue cell
3,epithelial cell,http://purl.obolibrary.org/obo/CL_0000066,http://purl.obolibrary.org/obo/CL_0000066,epithelial cell,http://purl.obolibrary.org/obo/CL_0000066,epithelial cell
4,blood vessel endothelial cell,http://purl.obolibrary.org/obo/CL_0000071,http://purl.obolibrary.org/obo/CL_0000115,endothelial cell,http://purl.obolibrary.org/obo/CL_0000115,endothelial cell
...,...,...,...,...,...,...
196,lung migratory dendritic cell,http://purl.obolibrary.org/obo/CL_4033045,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell
197,respiratory tract suprabasal cell,http://purl.obolibrary.org/obo/CL_4033048,http://purl.obolibrary.org/obo/CL_0000066,epithelial cell,http://purl.obolibrary.org/obo/CL_0000066,epithelial cell
198,cycling macrophage,http://purl.obolibrary.org/obo/CL_4033076,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell
199,cycling alveolar macrophage,http://purl.obolibrary.org/obo/CL_4033077,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell


## Add columns for `sc_transcriptomics` vs. `sc_proteomics`, CTann `tool`

In [175]:
# ctann tool
df_combined = df_cell_types_level_mapping.merge(
    df_crosswalks_combined,
    left_on='cell_label',
    right_on='CL_Label',
    how='left'
)

df_combined = df_combined.drop_duplicates(['cell_label','tool'])
df_combined

,cell_label,cell_id,level_1_cell_id,level_1_cell_label,level_2_cell_id,level_2_cell_label,CL_ID,CL_Label,tool
0,cell,http://purl.obolibrary.org/obo/CL_0000000,http://purl.obolibrary.org/obo/CL_0000000,no mapped parent cell,http://purl.obolibrary.org/obo/CL_0000000,no mapped parent cell,CL:0000000,cell,celltypist
1,hematopoietic stem cell,http://purl.obolibrary.org/obo/CL_0000037,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell,CL:0000037,hematopoietic stem cell,celltypist
2,hematopoietic stem cell,http://purl.obolibrary.org/obo/CL_0000037,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell,CL:0000037,hematopoietic stem cell,popv
7,hematopoietic stem cell,http://purl.obolibrary.org/obo/CL_0000037,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell,CL:0000037,hematopoietic stem cell,vccf
8,fibroblast,http://purl.obolibrary.org/obo/CL_0000057,http://purl.obolibrary.org/obo/CL_0002320,connective tissue cell,http://purl.obolibrary.org/obo/CL_0002320,connective tissue cell,CL:0000057,fibroblast,azimuth
...,...,...,...,...,...,...,...,...,...
1435,cycling macrophage,http://purl.obolibrary.org/obo/CL_4033076,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell,CL:4033076,cycling macrophage,azimuth
1436,cycling macrophage,http://purl.obolibrary.org/obo/CL_4033076,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell,CL:4033076,cycling macrophage,celltypist
1437,cycling alveolar macrophage,http://purl.obolibrary.org/obo/CL_4033077,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell,CL:4033077,cycling alveolar macrophage,azimuth
1439,cycling alveolar macrophage,http://purl.obolibrary.org/obo/CL_4033077,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell,CL:4033077,cycling alveolar macrophage,celltypist


In [176]:
# modality
df_combined['modality'] = df_combined['tool'].apply(lambda t: 'sc_transcriptomics' if t != 'vccf' else 'sc_proteomics')
df_combined

,cell_label,cell_id,level_1_cell_id,level_1_cell_label,level_2_cell_id,level_2_cell_label,CL_ID,CL_Label,tool,modality
0,cell,http://purl.obolibrary.org/obo/CL_0000000,http://purl.obolibrary.org/obo/CL_0000000,no mapped parent cell,http://purl.obolibrary.org/obo/CL_0000000,no mapped parent cell,CL:0000000,cell,celltypist,sc_transcriptomics
1,hematopoietic stem cell,http://purl.obolibrary.org/obo/CL_0000037,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell,CL:0000037,hematopoietic stem cell,celltypist,sc_transcriptomics
2,hematopoietic stem cell,http://purl.obolibrary.org/obo/CL_0000037,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell,CL:0000037,hematopoietic stem cell,popv,sc_transcriptomics
7,hematopoietic stem cell,http://purl.obolibrary.org/obo/CL_0000037,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell,CL:0000037,hematopoietic stem cell,vccf,sc_proteomics
8,fibroblast,http://purl.obolibrary.org/obo/CL_0000057,http://purl.obolibrary.org/obo/CL_0002320,connective tissue cell,http://purl.obolibrary.org/obo/CL_0002320,connective tissue cell,CL:0000057,fibroblast,azimuth,sc_transcriptomics
...,...,...,...,...,...,...,...,...,...,...
1435,cycling macrophage,http://purl.obolibrary.org/obo/CL_4033076,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell,CL:4033076,cycling macrophage,azimuth,sc_transcriptomics
1436,cycling macrophage,http://purl.obolibrary.org/obo/CL_4033076,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell,CL:4033076,cycling macrophage,celltypist,sc_transcriptomics
1437,cycling alveolar macrophage,http://purl.obolibrary.org/obo/CL_4033077,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell,CL:4033077,cycling alveolar macrophage,azimuth,sc_transcriptomics
1439,cycling alveolar macrophage,http://purl.obolibrary.org/obo/CL_4033077,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell,CL:4033077,cycling alveolar macrophage,celltypist,sc_transcriptomics


## Visualize

In [ ]:
# preprocess into correct format for ASCT+B Reporter, see https://docs.google.com/spreadsheets/d/1ISKJOktR6pl6uUNhLdV6xcXMZAQ3KfEgf8BMp9Jjs9s/edit?gid=0#gid=0
# Need: 
# AS/1	AS/1/LABEL	AS/1/ID (for cell)
# AS/2	AS/2/LABEL	AS/2/ID (for level 1)
# AS/3	AS/3/LABEL	AS/3/ID (for level 2)
# AS/4	AS/4/LABEL	AS/4/ID (for level 3)
# CT/1	CT/1/LABEL	CT/1/ID (for modality)
# BGene/1	BGene/1/LABEL	BGene/1/ID (for tools)

df_result = df_combined
 
df_result['AS/1'] = 'cell'
df_result['AS/1/LABEL'] = 'cell'
df_result['AS/1/ID'] = 'http://purl.obolibrary.org/obo/CL_0000000'

# AS/2	AS/2/LABEL	AS/2/ID (for level 1)
df_result = df_result.rename(columns=
  {
    'level_1_cell_label' : 'AS/2',
    'level_1_cell_id': 'AS/2/ID'
    }
)
df_result['AS/2/LABEL'] = df_result['AS/2']

# AS/3	AS/3/LABEL	AS/3/ID (for level 2)
df_result = df_result.rename(columns={
    'level_2_cell_label': 'AS/3',
    'level_2_cell_id': 'AS/3/ID'
}
)
df_result['AS/3/LABEL'] = df_result['AS/3']

# AS/4	AS/4/LABEL	AS/4/ID (for level 3)
df_result = df_result.rename(columns={
    'cell_label': 'AS/4',
    'cell_id': 'AS/4/ID'
}
)
df_result['AS/4/LABEL'] = df_result['AS/4']

# CT/1	CT/1/LABEL	CT/1/ID (for modality)
df_result = df_result.rename(columns={
    'tool': 'CT/1'
}
)
df_result['CT/1/LABEL'] = df_result['CT/1']
df_result['CT/1/ID'] = df_result['CT/1']

# BGene/1	BGene/1/LABEL	BGene/1/ID (for tools)
df_result = df_result.rename(columns={
    'modality': 'BGene/1'
}
)
df_result['BGene/1/LABEL'] = df_result['BGene/1']
df_result['BGene/1/ID'] = df_result['BGene/1']

df_result

,AS/4,AS/4/ID,AS/2/ID,AS/2,AS/3/ID,AS/3,CL_ID,CL_Label,CT/1,BGene/1,AS/1,AS/1/LABEL,AS/1/ID,AS/2/LABEL,AS/3/LABEL,AS/4/LABEL,CT/1/LABEL,CT/1/ID,BGene/1/LABEL,BGene/1/ID
0,cell,http://purl.obolibrary.org/obo/CL_0000000,http://purl.obolibrary.org/obo/CL_0000000,no mapped parent cell,http://purl.obolibrary.org/obo/CL_0000000,no mapped parent cell,CL:0000000,cell,celltypist,sc_transcriptomics,cell,cell,http://purl.obolibrary.org/obo/CL_0000000,no mapped parent cell,no mapped parent cell,cell,celltypist,celltypist,sc_transcriptomics,sc_transcriptomics
1,hematopoietic stem cell,http://purl.obolibrary.org/obo/CL_0000037,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell,CL:0000037,hematopoietic stem cell,celltypist,sc_transcriptomics,cell,cell,http://purl.obolibrary.org/obo/CL_0000000,hematopoietic cell,hematopoietic cell,hematopoietic stem cell,celltypist,celltypist,sc_transcriptomics,sc_transcriptomics
2,hematopoietic stem cell,http://purl.obolibrary.org/obo/CL_0000037,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell,CL:0000037,hematopoietic stem cell,popv,sc_transcriptomics,cell,cell,http://purl.obolibrary.org/obo/CL_0000000,hematopoietic cell,hematopoietic cell,hematopoietic stem cell,popv,popv,sc_transcriptomics,sc_transcriptomics
7,hematopoietic stem cell,http://purl.obolibrary.org/obo/CL_0000037,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell,CL:0000037,hematopoietic stem cell,vccf,sc_proteomics,cell,cell,http://purl.obolibrary.org/obo/CL_0000000,hematopoietic cell,hematopoietic cell,hematopoietic stem cell,vccf,vccf,sc_proteomics,sc_proteomics
8,fibroblast,http://purl.obolibrary.org/obo/CL_0000057,http://purl.obolibrary.org/obo/CL_0002320,connective tissue cell,http://purl.obolibrary.org/obo/CL_0002320,connective tissue cell,CL:0000057,fibroblast,azimuth,sc_transcriptomics,cell,cell,http://purl.obolibrary.org/obo/CL_0000000,connective tissue cell,connective tissue cell,fibroblast,azimuth,azimuth,sc_transcriptomics,sc_transcriptomics
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1435,cycling macrophage,http://purl.obolibrary.org/obo/CL_4033076,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell,CL:4033076,cycling macrophage,azimuth,sc_transcriptomics,cell,cell,http://purl.obolibrary.org/obo/CL_0000000,hematopoietic cell,hematopoietic cell,cycling macrophage,azimuth,azimuth,sc_transcriptomics,sc_transcriptomics
1436,cycling macrophage,http://purl.obolibrary.org/obo/CL_4033076,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell,CL:4033076,cycling macrophage,celltypist,sc_transcriptomics,cell,cell,http://purl.obolibrary.org/obo/CL_0000000,hematopoietic cell,hematopoietic cell,cycling macrophage,celltypist,celltypist,sc_transcriptomics,sc_transcriptomics
1437,cycling alveolar macrophage,http://purl.obolibrary.org/obo/CL_4033077,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell,CL:4033077,cycling alveolar macrophage,azimuth,sc_transcriptomics,cell,cell,http://purl.obolibrary.org/obo/CL_0000000,hematopoietic cell,hematopoietic cell,cycling alveolar macrophage,azimuth,azimuth,sc_transcriptomics,sc_transcriptomics
1439,cycling alveolar macrophage,http://purl.obolibrary.org/obo/CL_4033077,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell,CL:4033077,cycling alveolar macrophage,celltypist,sc_transcriptomics,cell,cell,http://purl.obolibrary.org/obo/CL_0000000,hematopoietic cell,hematopoietic cell,cycling alveolar macrophage,celltypist,celltypist,sc_transcriptomics,sc_transcriptomics


In [178]:
# order columns
# Define your desired column order
column_order = [
    'AS/1', 'AS/1/LABEL', 'AS/1/ID',
    'AS/2', 'AS/2/LABEL', 'AS/2/ID',
    'AS/3', 'AS/3/LABEL', 'AS/3/ID',
    'AS/4', 'AS/4/LABEL', 'AS/4/ID',
    'CT/1', 'CT/1/LABEL', 'CT/1/ID',
    'BGene/1','BGene/1/LABEL', 'BGene/1/ID'
]

# Reorder DataFrame columns
df_result = df_result[column_order]
df_result

,AS/1,AS/1/LABEL,AS/1/ID,AS/2,AS/2/LABEL,AS/2/ID,AS/3,AS/3/LABEL,AS/3/ID,AS/4,AS/4/LABEL,AS/4/ID,CT/1,CT/1/LABEL,CT/1/ID,BGene/1,BGene/1/LABEL,BGene/1/ID
0,cell,cell,http://purl.obolibrary.org/obo/CL_0000000,no mapped parent cell,no mapped parent cell,http://purl.obolibrary.org/obo/CL_0000000,no mapped parent cell,no mapped parent cell,http://purl.obolibrary.org/obo/CL_0000000,cell,cell,http://purl.obolibrary.org/obo/CL_0000000,celltypist,celltypist,celltypist,sc_transcriptomics,sc_transcriptomics,sc_transcriptomics
1,cell,cell,http://purl.obolibrary.org/obo/CL_0000000,hematopoietic cell,hematopoietic cell,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell,hematopoietic cell,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic stem cell,hematopoietic stem cell,http://purl.obolibrary.org/obo/CL_0000037,celltypist,celltypist,celltypist,sc_transcriptomics,sc_transcriptomics,sc_transcriptomics
2,cell,cell,http://purl.obolibrary.org/obo/CL_0000000,hematopoietic cell,hematopoietic cell,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell,hematopoietic cell,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic stem cell,hematopoietic stem cell,http://purl.obolibrary.org/obo/CL_0000037,popv,popv,popv,sc_transcriptomics,sc_transcriptomics,sc_transcriptomics
7,cell,cell,http://purl.obolibrary.org/obo/CL_0000000,hematopoietic cell,hematopoietic cell,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell,hematopoietic cell,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic stem cell,hematopoietic stem cell,http://purl.obolibrary.org/obo/CL_0000037,vccf,vccf,vccf,sc_proteomics,sc_proteomics,sc_proteomics
8,cell,cell,http://purl.obolibrary.org/obo/CL_0000000,connective tissue cell,connective tissue cell,http://purl.obolibrary.org/obo/CL_0002320,connective tissue cell,connective tissue cell,http://purl.obolibrary.org/obo/CL_0002320,fibroblast,fibroblast,http://purl.obolibrary.org/obo/CL_0000057,azimuth,azimuth,azimuth,sc_transcriptomics,sc_transcriptomics,sc_transcriptomics
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1435,cell,cell,http://purl.obolibrary.org/obo/CL_0000000,hematopoietic cell,hematopoietic cell,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell,hematopoietic cell,http://purl.obolibrary.org/obo/CL_0000988,cycling macrophage,cycling macrophage,http://purl.obolibrary.org/obo/CL_4033076,azimuth,azimuth,azimuth,sc_transcriptomics,sc_transcriptomics,sc_transcriptomics
1436,cell,cell,http://purl.obolibrary.org/obo/CL_0000000,hematopoietic cell,hematopoietic cell,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell,hematopoietic cell,http://purl.obolibrary.org/obo/CL_0000988,cycling macrophage,cycling macrophage,http://purl.obolibrary.org/obo/CL_4033076,celltypist,celltypist,celltypist,sc_transcriptomics,sc_transcriptomics,sc_transcriptomics
1437,cell,cell,http://purl.obolibrary.org/obo/CL_0000000,hematopoietic cell,hematopoietic cell,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell,hematopoietic cell,http://purl.obolibrary.org/obo/CL_0000988,cycling alveolar macrophage,cycling alveolar macrophage,http://purl.obolibrary.org/obo/CL_4033077,azimuth,azimuth,azimuth,sc_transcriptomics,sc_transcriptomics,sc_transcriptomics
1439,cell,cell,http://purl.obolibrary.org/obo/CL_0000000,hematopoietic cell,hematopoietic cell,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell,hematopoietic cell,http://purl.obolibrary.org/obo/CL_0000988,cycling alveolar macrophage,cycling alveolar macrophage,http://purl.obolibrary.org/obo/CL_4033077,celltypist,celltypist,celltypist,sc_transcriptomics,sc_transcriptomics,sc_transcriptomics


## Export

In [179]:
df_result.to_csv('output/ctann_tree.csv', index=False)
# put on Google Sheets: https://docs.google.com/spreadsheets/d/1ISKJOktR6pl6uUNhLdV6xcXMZAQ3KfEgf8BMp9Jjs9s/edit?gid=1266919613#gid=1266919613